In [ ]:
# 01_Python_code/post_processing/plot_best_models_comparison.py
"""
Generate 48 line plots comparing best models per group.

For each combination of (block × direction × price_type) creates:
  - 24 plots: best models selected by COMBINED_SCORE
  - 24 plots: best models selected by REVENUE

Each plot contains 4 lines:
    1. Actual value
    2. Best statistical model forecast (STATISTICAL_EXPANDING or STATISTICAL_ROLLING)
    3. Best LSTM forecast (DEEP_LEARNING)
    4. Best XGBoost forecast (TREE_BASED)

Output folders:
    03_Output_paper/plots/best_models_comparison/combined_score/
    03_Output_paper/plots/best_models_comparison/revenue/
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ─── Paths ────────────────────────────────────────────────────────────────────

OUTPUT_PAPER  = Path(r"c:\Users\A80391\E.ON\my_tasks\00_personal\Economic_LSTM\03_Output_paper")
POSTPROCESSED = OUTPUT_PAPER / "postprocessed_data"
PLOTS_OUT     = OUTPUT_PAPER / "plots" / "best_models_comparison"

# ─── Model registry: MODEL_BASE → (folder_name, file_prefix) ─────────────────

MODEL_REGISTRY = {
    # Statistical – expanding window
    "arma":              ("arma_model_output",              "arma"),
    "armax_exog_1":      ("armax_exog_1_model_output",      "armax_exog_1"),
    "armax_exog_4":      ("armax_exog_4_model_output",      "armax_exog_4"),
    "armax_exog_1_3":    ("armax_exog_1_3_model_output",    "armax_exog_1_3"),
    "armax_exog_1_4":    ("armax_exog_1_4_model_output",    "armax_exog_1_4"),
    "sarma":             ("sarma_model_output",             "sarma"),
    "sarmax_exog_1":     ("sarmax_exog_1_model_output",     "sarmax_exog_1"),
    "sarmax_exog_4":     ("sarmax_exog_4_model_output",     "sarmax_exog_4"),
    "sarmax_exog_1_3":   ("sarmax_exog_1_3_model_output",   "sarmax_exog_1_3"),
    "sarmax_exog_1_4":   ("sarmax_exog_1_4_model_output",   "sarmax_exog_1_4"),
    # Statistical – rolling window
    "rw_arma_3m":              ("rw_arma_3m_model_output",              "rw_arma_3m"),
    "rw_arma_6m":              ("rw_arma_6m_model_output",              "rw_arma_6m"),
    "rw_arma_12m":             ("rw_arma_12m_model_output",             "rw_arma_12m"),
    "rw_armax_exog_1_3m":      ("rw_armax_exog_1_3m_model_output",      "rw_armax_exog_1_3m"),
    "rw_armax_exog_1_6m":      ("rw_armax_exog_1_6m_model_output",      "rw_armax_exog_1_6m"),
    "rw_armax_exog_1_12m":     ("rw_armax_exog_1_12m_model_output",     "rw_armax_exog_1_12m"),
    "rw_armax_exog_4_3m":      ("rw_armax_exog_4_3m_model_output",      "rw_armax_exog_4_3m"),
    "rw_armax_exog_4_6m":      ("rw_armax_exog_4_6m_model_output",      "rw_armax_exog_4_6m"),
    "rw_armax_exog_4_12m":     ("rw_armax_exog_4_12m_model_output",     "rw_armax_exog_4_12m"),
    "rw_armax_exog_1_3_3m":    ("rw_armax_exog_1_3_3m_model_output",    "rw_armax_exog_1_3_3m"),
    "rw_armax_exog_1_3_6m":    ("rw_armax_exog_1_3_6m_model_output",    "rw_armax_exog_1_3_6m"),
    "rw_armax_exog_1_3_12m":   ("rw_armax_exog_1_3_12m_model_output",   "rw_armax_exog_1_3_12m"),
    "rw_armax_exog_1_4_3m":    ("rw_armax_exog_1_4_3m_model_output",    "rw_armax_exog_1_4_3m"),
    "rw_armax_exog_1_4_6m":    ("rw_armax_exog_1_4_6m_model_output",    "rw_armax_exog_1_4_6m"),
    "rw_armax_exog_1_4_12m":   ("rw_armax_exog_1_4_12m_model_output",   "rw_armax_exog_1_4_12m"),
    "rw_sarma_3m":             ("rw_sarma_3m_model_output",             "rw_sarma_3m"),
    "rw_sarma_6m":             ("rw_sarma_6m_model_output",             "rw_sarma_6m"),
    "rw_sarma_12m":            ("rw_sarma_12m_model_output",            "rw_sarma_12m"),
    "rw_sarmax_exog_1_3m":     ("rw_sarmax_exog_1_3m_model_output",     "rw_sarmax_exog_1_3m"),
    "rw_sarmax_exog_1_6m":     ("rw_sarmax_exog_1_6m_model_output",     "rw_sarmax_exog_1_6m"),
    "rw_sarmax_exog_1_12m":    ("rw_sarmax_exog_1_12m_model_output",    "rw_sarmax_exog_1_12m"),
    "rw_sarmax_exog_4_3m":     ("rw_sarmax_exog_4_3m_model_output",     "rw_sarmax_exog_4_3m"),
    "rw_sarmax_exog_4_6m":     ("rw_sarmax_exog_4_6m_model_output",     "rw_sarmax_exog_4_6m"),
    "rw_sarmax_exog_4_12m":    ("rw_sarmax_exog_4_12m_model_output",    "rw_sarmax_exog_4_12m"),
    "rw_sarmax_exog_1_3_3m":   ("rw_sarmax_exog_1_3_3m_model_output",   "rw_sarmax_exog_1_3_3m"),
    "rw_sarmax_exog_1_3_6m":   ("rw_sarmax_exog_1_3_6m_model_output",   "rw_sarmax_exog_1_3_6m"),
    "rw_sarmax_exog_1_3_12m":  ("rw_sarmax_exog_1_3_12m_model_output",  "rw_sarmax_exog_1_3_12m"),
    "rw_sarmax_exog_1_4_3m":   ("rw_sarmax_exog_1_4_3m_model_output",   "rw_sarmax_exog_1_4_3m"),
    "rw_sarmax_exog_1_4_6m":   ("rw_sarmax_exog_1_4_6m_model_output",   "rw_sarmax_exog_1_4_6m"),
    "rw_sarmax_exog_1_4_12m":  ("rw_sarmax_exog_1_4_12m_model_output",  "rw_sarmax_exog_1_4_12m"),
    # Deep learning (LSTM family)
    "lstm":                ("lstm_model_output",             "lstm"),
    "bidirectional_lstm":  ("bidirectional_lstm_model_output", "bidirectional_lstm"),
    "lstm_features_1":     ("lstm_features_1_model_output",  "lstm_features_1"),
    "lstm_features_4":     ("lstm_features_4_model_output",  "lstm_features_4"),
    "lstm_features_1_3":   ("lstm_features_1_3_model_output","lstm_features_1_3"),
    "lstm_features_1_4":   ("lstm_features_1_4_model_output","lstm_features_1_4"),
    # Tree-based
    "xgboost_exog_4":      ("xgboost_exog_4_model_output",    "xgboost_exog_4"),
    "xgboost_v2_exog_1":   ("xgboost_v2_exog_1_model_output", "xgboost_v2_exog_1"),
    "xgboost_v2_exog_4":   ("xgboost_v2_exog_4_model_output", "xgboost_v2_exog_4"),
    "xgboost_v2_exog_1_3": ("xgboost_v2_exog_1_3_model_output","xgboost_v2_exog_1_3"),
    "xgboost_v2_exog_1_4": ("xgboost_v2_exog_1_4_model_output","xgboost_v2_exog_1_4"),
}

# LSTM bases use multi-column result files
LSTM_BASES = {
    "lstm", "bidirectional_lstm",
    "lstm_features_1", "lstm_features_4",
    "lstm_features_1_3", "lstm_features_1_4",
}

STRATEGY_TO_COLUMN = {
    "ensemble": "D+1_ENSEMBLE_MODEL",
    "recency":  "D+1_RECENCY_WEIGHTED",
    "single":   "D+1_SINGLE_MODEL",
}

STATISTICAL_GROUPS = {"STATISTICAL_EXPANDING", "STATISTICAL_ROLLING"}

# ─── File-path builder ────────────────────────────────────────────────────────

def _build_file_path(model_base: str, lookback, strategy,
                     product: str, sign: str, ptype: str) -> Path:
    """Return the expected result CSV path for one model row."""
    if model_base not in MODEL_REGISTRY:
        raise KeyError(f"Unknown MODEL_BASE: '{model_base}'")

    folder, prefix = MODEL_REGISTRY[model_base]
    folder_path = OUTPUT_PAPER / folder

    # XGBoost v2 → filename includes ts{lookback}
    if model_base.startswith("xgboost_v2_exog_"):
        lb = int(lookback)
        fname = f"{prefix}_ts{lb}_model_results_{product}_{sign}_{ptype}.csv"

    # XGBoost v1 → no lookback in filename
    elif model_base.startswith("xgboost_exog_"):
        fname = f"{prefix}_model_results_{product}_{sign}_{ptype}.csv"

    # LSTM family → filename includes ts{lookback}
    elif model_base in LSTM_BASES:
        lb = int(lookback)
        fname = f"{prefix}_ts{lb}_model_results_{product}_{sign}_{ptype}.csv"

    # Statistical (expanding or rolling) – no lookback in filename
    else:
        fname = f"{prefix}_model_results_{product}_{sign}_{ptype}.csv"

    return folder_path / fname


def _load_series(row: pd.Series,
                 product: str, sign: str, ptype: str):
    """
    Return (actual_series, forecast_series) for one model row.
    Both series are indexed by DATE.
    """
    base     = row["MODEL_BASE"]
    lookback = row.get("LOOKBACK")
    strategy = row.get("STRATEGY")

    path = _build_file_path(base, lookback, strategy, product, sign, ptype)
    if not path.exists():
        raise FileNotFoundError(f"Missing result file: {path}")

    df = pd.read_csv(path, parse_dates=["DATE"], index_col="DATE")
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    if "ACTUAL_VALUE" not in df.columns:
        raise KeyError(f"'ACTUAL_VALUE' missing in {path}")

    if base in LSTM_BASES:
        if pd.isna(strategy) or strategy not in STRATEGY_TO_COLUMN:
            raise ValueError(f"Invalid LSTM strategy '{strategy}' for {base}")
        fc_col = STRATEGY_TO_COLUMN[strategy]
        if fc_col not in df.columns:
            raise KeyError(f"Column '{fc_col}' not found in {path}")
        return df["ACTUAL_VALUE"], df[fc_col]
    else:
        if "D+1" not in df.columns:
            raise KeyError(f"'D+1' missing in {path}")
        return df["ACTUAL_VALUE"], df["D+1"]


# ─── Best model selection ─────────────────────────────────────────────────────

def _find_best(
    df: pd.DataFrame,
    product: str,
    sign: str,
    ptype: str,
    score_col: str,
):
    """
    For one (product, sign, ptype) combo return the best-scoring row
    from each of the three groups: statistical, LSTM, XGBoost.
    """
    mask = (
        (df["PRODUCT"] == product)
        & (df["PRODUCT_SIGN"] == sign)
        & (df["PRICE_TYPE"] == ptype)
    )
    subset = df[mask].copy()

    def _best(allowed_groups):
        rows = subset[subset["MODEL_GROUP"].isin(allowed_groups)].dropna(subset=[score_col])
        if rows.empty:
            return None
        return rows.loc[rows[score_col].idxmax()]

    return (
        _best(STATISTICAL_GROUPS),
        _best({"DEEP_LEARNING"}),
        _best({"TREE_BASED"}),
    )


# ─── Plotting ─────────────────────────────────────────────────────────────────

COLORS = {
    "actual": "#000000",
    "stat":   "#1f77b4",
    "lstm":   "#d62728",
    "xgb":   "#2ca02c",
}
LINESTYLES = {
    "actual": "-",
    "stat":   "--",
    "lstm":   "-.",
    "xgb":   ":",
}


def _align_series(*series_list):
    """
    Keep only dates that are present in ALL non-None series.
    Returns the same series filtered to the common index.
    """
    valid = [s for s in series_list if s is not None]
    if not valid:
        return series_list
    common_idx = valid[0].index
    for s in valid[1:]:
        common_idx = common_idx.intersection(s.index)
    return tuple(s.loc[common_idx] if s is not None else None for s in series_list)


import seaborn as sns

def _make_plot(
    actual:    pd.Series,
    stat_fc:   pd.Series | None,
    lstm_fc:   pd.Series | None,
    xgb_fc:    pd.Series | None,
    stat_name: str,
    lstm_name: str,
    xgb_name:  str,
    title:     str,
    out_path:  Path,
) -> None:

    # Align all series to common dates
    actual, stat_fc, lstm_fc, xgb_fc = _align_series(actual, stat_fc, lstm_fc, xgb_fc)

    # ── Filter to start date ──────────────────────────────────────────────────
    start_date = pd.Timestamp("2024-12-15")
    actual = actual.loc[actual.index >= start_date]
    if stat_fc is not None: stat_fc = stat_fc.loc[stat_fc.index >= start_date]
    if lstm_fc is not None: lstm_fc = lstm_fc.loc[lstm_fc.index >= start_date]
    if xgb_fc  is not None: xgb_fc  = xgb_fc.loc[xgb_fc.index  >= start_date]

    # ── Compute differences (forecast − actual) ───────────────────────────────
    stat_diff = stat_fc - actual if stat_fc is not None else None
    lstm_diff = lstm_fc - actual if lstm_fc is not None else None
    xgb_diff  = xgb_fc  - actual if xgb_fc  is not None else None

    sns.set_theme(style="whitegrid", font_scale=0.9)
    fig, ax = plt.subplots(figsize=(15, 5))

    # Zero reference line
    ax.axhline(0, color=COLORS["actual"], lw=1.5, ls=LINESTYLES["actual"],
               label="Actual (baseline)", zorder=5)

    if stat_diff is not None:
        sns.lineplot(x=stat_diff.index, y=stat_diff.values, ax=ax,
                     color=COLORS["stat"], linestyle=LINESTYLES["stat"],
                     linewidth=1.5, label=f"Statistical: {stat_name}", zorder=4)

    if lstm_diff is not None:
        sns.lineplot(x=lstm_diff.index, y=lstm_diff.values, ax=ax,
                     color=COLORS["lstm"], linestyle=LINESTYLES["lstm"],
                     linewidth=1.5, label=f"LSTM: {lstm_name}", zorder=4)

    if xgb_diff is not None:
        sns.lineplot(x=xgb_diff.index, y=xgb_diff.values, ax=ax,
                     color=COLORS["xgb"], linestyle=LINESTYLES["xgb"],
                     linewidth=1.5, label=f"XGBoost: {xgb_name}", zorder=4)

    # ── Weekly x-axis ticks ───────────────────────────────────────────────────
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%#d %b '%y"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=40, ha="right", fontsize=8)

    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Date", fontsize=10)
    ax.set_ylabel("Forecast − Actual [EUR/MW/h]", fontsize=10)
    ax.legend(loc="lower left", fontsize=8, framealpha=0.85, ncol=2)

    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out_path.name}")


# ─── Main ─────────────────────────────────────────────────────────────────────

def main() -> None:
    print("Loading all_models_comparison.csv …")
    df = pd.read_csv(POSTPROCESSED / "all_models_comparison.csv")

    products   = df["PRODUCT"].unique()
    signs      = df["PRODUCT_SIGN"].unique()
    ptypes     = df["PRICE_TYPE"].unique()

    for score_col, sub_folder in [("COMBINED_SCORE", "combined_score"), ("REVENUE", "revenue")]:
        out_dir = PLOTS_OUT / sub_folder
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n── Generating plots for score: {score_col} → {out_dir}")

        for product in products:
            for sign in signs:
                for ptype in ptypes:
                    stat_row, lstm_row, xgb_row = _find_best(df, product, sign, ptype, score_col)

                    if stat_row is None and lstm_row is None and xgb_row is None:
                        print(f"  Skipping {product}_{sign}_{ptype}: no models found.")
                        continue

                    # Load actual from the first available row
                    actual = None
                    stat_fc = lstm_fc = xgb_fc = None
                    stat_name = lstm_name = xgb_name = "N/A"

                    for row, key in [(stat_row, "stat"), (lstm_row, "lstm"), (xgb_row, "xgb")]:
                        if row is None:
                            continue
                        try:
                            act, fc = _load_series(row, product, sign, ptype)
                            if actual is None:
                                actual = act
                            if key == "stat":
                                stat_fc   = fc
                                stat_name = row["MODEL_BASE"]
                            elif key == "lstm":
                                lstm_fc   = fc
                                lstm_name = f"{row['MODEL_BASE']} ({row.get('STRATEGY', '')})"
                            elif key == "xgb":
                                xgb_fc   = fc
                                xgb_name = row["MODEL_BASE"]
                        except (FileNotFoundError, KeyError, ValueError) as e:
                            print(f"  WARNING: {e}")

                    if actual is None:
                        print(f"  Skipping {product}_{sign}_{ptype}: could not load actual series.")
                        continue

                    title    = f"{product} | {sign} | {ptype} — best models by {score_col}"
                    out_path = out_dir / f"{product}_{sign}_{ptype}.png"

                    _make_plot(
                        actual=actual,
                        stat_fc=stat_fc,
                        lstm_fc=lstm_fc,
                        xgb_fc=xgb_fc,
                        stat_name=stat_name,
                        lstm_name=lstm_name,
                        xgb_name=xgb_name,
                        title=title,
                        out_path=out_path,
                    )

    print("\nDone.")


if __name__ == "__main__":
    main()
